# Setup inicial da base — bootstrap (rodar 1 vez no banco)

**Pode dar `Run All`** — nada aborta o notebook: cada bloco mostra `[OK]`/`[FALHA]` e segue. Ao final, a célula de conferência mostra as contagens. Se algum bloco der `[FALHA]`, o erro fica **só nele** — corrija e **re-rode só ele** (todos são idempotentes).

⚠️ **É demorado** (Anbima Data completa + boletim de ~4 meses + cálculo dia a dia hitando APIs — pode levar horas). As janelas já vêm no máximo que cada fonte entrega.

**Rode a partir da pasta `code/`.** Depois do setup, o dia a dia é o outro notebook: `pipeline.ipynb`. Ver `vault/13 - Migracao Banco.md` §5 e `vault/11 - Pipeline de Execucao.md`.

## Config (rodar primeiro) — janelas máximas por fonte

In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

scripts = Path.cwd() / "scripts"
if not scripts.exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(scripts))
import pipeline_core as pc

HOJE        = date.today().isoformat()
INI_IND     = (date.today() - timedelta(days=130)).isoformat()  # deb/NTN-B: fonte guarda ~4 meses
INI_BOLETIM = INI_IND       # boletim alinhado a janela de referencia (estenda se quiser mais historico)
DIAS_DI     = 20            # curva DI B3: fonte guarda ~20 pregoes
DIAS_CRICRA = 5            # CRI/CRA: portal guarda ~5 pregoes

DIAS = [d.isoformat() for d in pc.dias_uteis_entre(INI_BOLETIM, HOJE)]
print("HOJE:", HOJE, "| INI_IND:", INI_IND, "| INI_BOLETIM:", INI_BOLETIM)
print("dias uteis na janela do boletim:", len(DIAS))

## Scraping — 1 bloco por fonte
Cada bloco já vem na janela máxima. Se um falhar (proxy, Playwright, login, Bloomberg), corrija e re-rode só ele.

In [ ]:
# 1. Anbima Data (caracteristicas + fluxo) — universo COMPLETO. O mais pesado (~min).
pc.anbima_data(full=True)

In [ ]:
# 2. FI Analytics planilha (caracteristicas) — snapshot atual
pc.fianalytics()

In [ ]:
# 3. Anbima debentures (indicativas) — ~4 meses (fonte skipa 404 fora da janela)
pc.anbima_deb(INI_IND, HOJE)

In [ ]:
# 4. Anbima NTN-B (MtM) — ~4 meses
pc.ntnb(INI_IND, HOJE)

In [ ]:
# 5. Curva DI B3 (MtM) — ultimos ~20 pregoes (limite da fonte)
for d in pc.ultimos_n_dias_uteis(DIAS_DI):
    pc.curva_di(d)

In [ ]:
# 6. Anbima CRI/CRA (indicativas) — ultimos ~5 pregoes (limite do portal)
for d in pc.ultimos_n_dias_uteis(DIAS_CRICRA):
    pc.anbima_cricra(d)

In [ ]:
# 7. Boletim B3 (negocios) — desde INI_BOLETIM
pc.boletim(INI_BOLETIM, HOJE)

In [ ]:
# 8. Outstanding via Bloomberg — SO NO BANCO (no PC pessoal da [FALHA], tudo bem)
pc.outstanding(INI_BOLETIM, HOJE)

## Cálculo — percorre todos os pregões da janela (`DIAS`)
Idempotentes: re-rodar pula o já calculado. Ordem: taxa → filtrar → spread Anbima → match → spread over → relatório.

In [ ]:
# 9. Calcular taxa por trade (cascata FI Analytics -> B3)
for X in DIAS:
    pc.calc_taxa(X)

In [ ]:
# 10. Filtrar (VALIDO / FUNDO / BROKER / PF)
for X in DIAS:
    pc.filtrar(X)

In [ ]:
# 11. Spread Anbima das indicativas
for X in DIAS:
    pc.spread_anbima(X)

In [ ]:
# 12. Match de referencia (global, sem data)
pc.match_ref()

In [ ]:
# 13. Spread over dos trades
for X in DIAS:
    pc.spread_over(X)

In [ ]:
# 14. Gerar relatorio (toda a base)
pc.relatorio()

## Conferência — a base foi montada?
Roda junto no Run All. Contagens > 0 nas tabelas principais = deu certo. Se algo ficou zerado, veja qual bloco acima deu `[FALHA]` e re-rode só ele (+ os blocos de cálculo).

In [ ]:
import sqlite3
c = sqlite3.connect('data/trades.db')
for t in ['NegociosBrutos','NegociosProcessados','InfoAtivos','AnbimaIndicativos','MtmAnbima','FluxoAtivos','Outstanding']:
    print(f'  {t:22s}: {c.execute("SELECT COUNT(*) FROM "+t).fetchone()[0]:>9,} linhas')
c.close()